# Chapter 13: Agentic RAG
## Topic 3: "Check the Catalog Only If Uncertain" Decision Step

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- Topic 2 established confidence-based retrieval triggering as a general principle. This topic takes that principle and applies it to one, specific, concrete case — `lookup_product_catalog` (Chapter 10 Topic 6) — building the actual decision step a real agent would execute, rather than continuing to reason about confidence-based triggering only in the abstract.
- Why `lookup_product_catalog` specifically, rather than `search_knowledge_base` (Chapter 9 Topic 3) or `validate_fd_reference` (Chapter 10 Topic 3): product-specific facts (an exact interest rate, tenure, or minimum deposit for a *named* product) are the cleanest possible case of Topic 2's confidence principle, because a model's *correct* behavior is almost always "uncertain" for genuinely specific product terms — unlike general FD policy, where a model might legitimately, correctly be confident about well-established, stable facts. This makes the catalog-lookup decision a sharper, more clear-cut test of the underlying pattern than a more nuanced, general-knowledge case would be.
- The core mechanism this topic builds concretely: given a customer query that mentions a specific product name (or something that might be one), the agent must decide — before answering — whether it's confident it already knows this product's exact terms, or whether it needs to call `lookup_product_catalog` first. This is a single, specific instance of Topic 2's general confidence-based triggering pattern, built out completely, end to end, for one clean, well-defined case.


### 2. Internal Working — Step by Step

**The "check only if uncertain" decision step, built out concretely:**

1. **Detect that the query references a specific, named product** — this is a lexical/pattern-matching step (a product name appearing in the query text), distinct from the confidence judgment itself; detecting *that* a product is mentioned is a simpler, more mechanical step than judging whether the model is confident about that product's exact terms.
2. **Apply the confidence rule specifically: for any genuinely named, specific product, the model should treat itself as uncertain by default**, rather than attempting to judge confidence case by case for each individual product. This is a deliberate simplification of Topic 2's more general confidence principle: rather than asking the model to introspect freshly for every product name, this topic's pattern encodes a blanket rule — named products always warrant a catalog check — precisely because Chapter 9 Topic 8's Smart Saver FD argument establishes that a model's training data cannot reliably be assumed to include any given specific product's terms, especially newly-launched ones.
3. **This blanket "always uncertain for named products" rule is a deliberate, evidence-based simplification, not an oversight** — Topic 2 discussed the general difficulty of validating a model's self-reported confidence; for the specific, narrow case of named-product terms, this topic sidesteps that difficulty entirely by not relying on the model's self-assessed confidence at all, instead encoding the trigger condition as a simpler, more reliable, pattern-based rule: "a specific product is named → check the catalog," full stop, regardless of how confident the model might otherwise feel.
4. **The lookup itself proceeds exactly as `lookup_product_catalog` was designed** (Chapter 10 Topic 6's structured, found/not-found-shaped output) — this topic adds no new retrieval mechanism, only the specific, narrow decision rule governing when this particular tool gets called.


### 3. How This Is Implemented in This Project

- The system prompt's instruction for this specific pattern can be considerably more direct and less hedged than Topic 2's general confidence-framing guidance: "if the customer's email mentions a specific, named FD product, always call `lookup_product_catalog` before stating any of that product's terms — never rely on general knowledge for a specific product's exact figures." This is a stronger, more absolute instruction than Topic 2's general "retrieve when uncertain" framing, justified specifically because product-name-triggered uncertainty is close to universal, not merely common.
- This pattern directly reuses Chapter 10 Topic 6's `lookup_product_catalog` tool and its existing schema (Chapter 10 Topic 4's principles) without modification — this topic's contribution is entirely about *when* the tool gets called (the decision step), not the tool's own design, which was already completed and validated in earlier chapters.
- This specific, narrow pattern is exactly what Topic 4's Smart Saver FD test validates end to end: a query mentioning "Smart Saver FD" by name should reliably trigger this decision step's rule, regardless of how the model might otherwise feel about its own general FD knowledge — Topic 4's test is this pattern's most direct, concrete validation.


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **Detecting that a query "mentions a specific, named product" is itself not always trivial** — a customer might reference a product by a partial name, a common misspelling, or an informal shorthand rather than its exact catalog name, meaning the pattern-detection step (distinct from the confidence-judgment step) has its own real failure modes, directly connecting to Chapter 9 Topic 2's Hinglish/vocabulary-mismatch discussion, now applied to product-name detection specifically rather than general query vocabulary.
- **The blanket "always uncertain for named products" rule trades away some potential efficiency for reliability** — Topic 2's more nuanced, case-by-case confidence framing could in principle allow confident answers for extremely well-established, unchanging product terms; this topic's simpler blanket rule always checks, accepting a small, bounded extra cost (one lookup call) in exchange for eliminating the harder, less reliable problem of judging per-product confidence individually.
- **This pattern's reliability is capped by the underlying `lookup_product_catalog` tool's own found/not-found correctness** (Chapter 10 Topic 6) — if a genuinely valid product name isn't recognized by the catalog lookup due to a formatting mismatch, this decision step correctly triggers the call, but the call itself then fails to find real, existing data — a downstream tool-implementation issue, not a flaw in this topic's triggering decision itself, requiring the same layered-attribution discipline (is the failure in the decision to call, or in the called tool's own logic) established throughout this notebook series.
- **Monitoring:** tracking how often `lookup_product_catalog` is called with a product name that the catalog then reports as `exists: False` reveals either a genuine gap in the catalog (a real product missing from it) or a name-matching problem (Chapter 10 Topic 6's exact-match design) — either way, a directly measurable, actionable signal distinct from whether the triggering *decision* itself was correct.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **A blanket rule (always check for named products) vs a nuanced, per-product confidence judgment (Topic 2's more general approach):** the blanket rule is simpler, more reliable, and easier to validate (a single pattern-detection test, rather than a harder confidence-calibration test), at the cost of occasionally triggering a lookup for a product whose terms the model might have genuinely, correctly known — a small, bounded, acceptable cost given the much larger risk (Chapter 9 Topic 8's argument) of confidently hallucinating a specific product's terms incorrectly.
- **How to handle a query that plausibly references a product but with ambiguous or uncertain matching** (a partial name, a likely misspelling): a conservative design triggers the catalog check anyway whenever there's real ambiguity about whether a specific product is being referenced, since the cost of an unnecessary lookup (Chapter 10 Topic 3's negligible-cost profile) is far smaller than the cost of a missed, genuinely-needed lookup.
- **Whether this pattern should extend to other tools beyond `lookup_product_catalog`:** the same "always trigger, don't rely on self-assessed confidence" simplification could apply to any tool where the trigger condition is reliably detectable independent of the model's own introspection (a specific reference number pattern for `validate_fd_reference`, for example) — worth applying this same design pattern wherever a similarly reliable, simpler trigger condition exists, rather than defaulting to Topic 2's more general (and harder-to-validate) confidence framing everywhere.


### 6. Alternatives and When to Use Each

- **Blanket "always check for named products" rule (this topic's approach):** the right choice specifically when the trigger condition (a named product being mentioned) is reliably, mechanically detectable, and the cost asymmetry strongly favors an occasional unnecessary check over a missed necessary one — exactly `lookup_product_catalog`'s situation.
- **Topic 2's more general, nuanced confidence framing:** appropriate for cases where the trigger condition isn't as mechanically detectable, or where the cost asymmetry is less strongly one-sided — general FD policy questions, where the model's confidence genuinely does vary meaningfully case by case.
- **Never triggering a catalog check, relying entirely on the model's general knowledge:** not appropriate for any project with a real, evolving product catalog, given Chapter 9 Topic 8's demonstrated argument that a model cannot reliably know a specific, possibly newly-launched product's terms from training data alone.


### 7. Common Mistakes and Production Failures

- Relying on the model's own case-by-case confidence judgment for named products, rather than encoding a simpler, more reliable blanket rule — reintroducing the harder confidence-calibration validation problem for a case where it's specifically avoidable.
- Not accounting for informal, partial, or misspelled product-name references when detecting whether a query mentions a specific product, missing genuine trigger cases due to a too-narrow pattern-matching step.
- Conflating a triggering-decision failure (the model didn't call the tool when it should have) with a tool-implementation failure (the tool was called correctly but the catalog lookup itself failed to find a genuinely existing product), applying the wrong fix to the wrong layer.
- Applying this topic's blanket-rule simplification to cases where it doesn't actually fit — trigger conditions that aren't reliably, mechanically detectable independent of genuine model judgment — rather than recognizing when Topic 2's more general, nuanced confidence framing is actually the more appropriate pattern.


### 8. Lead-Level Interview Questions

**Basic**

- Q: Why does this pattern use a blanket "always check" rule for named products rather than Topic 2's more nuanced, case-by-case confidence judgment?
  A: Product-specific terms are close to universally uncertain for a model — a specific product's exact interest rate, tenure, or minimum deposit is essentially never reliably knowable from general training data, especially for newly-launched products (Chapter 9 Topic 8's argument). This makes a simple, blanket rule ("named product mentioned → always check") both simpler to implement and more reliable than asking the model to judge its own confidence per product, sidestepping the harder confidence-calibration validation problem Topic 2 raised for less clear-cut cases.

- Q: What are the two genuinely separate steps in this decision pattern?
  A: First, detecting that the query mentions a specific, named product at all — a mechanical, pattern-matching step. Second, applying the trigger rule itself — for this topic's pattern, simply "if a named product was detected, always call `lookup_product_catalog`," without any further confidence judgment needed.

**Intermediate**

- Q: What's the trade-off this topic's blanket rule accepts, compared to a more nuanced confidence-based approach?
  A: It occasionally triggers a catalog lookup for a product whose terms the model might have genuinely, correctly known already — a small, bounded extra cost (one additional lookup call). This is accepted because the alternative risk — confidently hallucinating a specific product's terms incorrectly — is judged far more costly than an occasional unnecessary lookup, given `lookup_product_catalog`'s negligible per-call cost (Chapter 10 Topic 3).

- Q: How does this topic's pattern relate to the underlying `lookup_product_catalog` tool built in Chapter 10 Topic 6?
  A: This topic adds no new retrieval mechanism — it reuses `lookup_product_catalog` exactly as designed, contributing only the specific decision rule governing *when* that tool gets called (whenever a named product is detected in the query), leaving the tool's own internal design (structured output, exists/not-exists distinction) completely unchanged.

**Advanced**

- Q: Design the detection step for "the query mentions a specific, named product," accounting for informal or partial references.
  A: A pure exact-string-match against the catalog's product names would miss informal references, misspellings, or partial mentions — a more robust detection step should account for these, perhaps using a broader matching approach (checking for partial name overlap, or common informal shorthand) while erring toward triggering the catalog check whenever there's genuine ambiguity about whether a specific product is being referenced, since the cost asymmetry (a cheap unnecessary check vs. a costly missed one) favors over-triggering in ambiguous cases rather than under-triggering.

- Q: A teammate argues this blanket rule is wasteful, since many named-product questions could be answered from general knowledge (e.g., "does the Smart Saver FD allow premature withdrawal" might follow the same general policy as all other FDs). How do you respond?
  A: This is a real, legitimate distinction, but it's actually a *different* question than the one this pattern's blanket rule addresses — the blanket rule specifically governs claiming a product's *specific, exact figures* (its rate, tenure, minimum deposit), not general policy questions that happen to mention the product's name. A more refined version of this pattern could distinguish "asking about this named product's specific terms" (always check) from "asking a general policy question that happens to mention this product's name" (may not need a specific catalog check) — this refinement is a reasonable evolution of the pattern, but should be validated with real measured data (following Chapter 10 Topic 7's methodology) showing it doesn't reintroduce the specific-figure hallucination risk the current, simpler blanket rule is designed to eliminate.

**Scenario-based**

- Q: Production monitoring shows `lookup_product_catalog` is frequently called with a product name that returns `exists: False`. Walk through your diagnosis.
  A: This could mean either a genuine gap in the catalog (a real, existing product that was never added to it) or a name-matching problem (Chapter 10 Topic 6's exact-match design failing to recognize a valid product due to a formatting or spelling variation). Check whether the specific product names triggering this pattern actually exist in the underlying catalog data at all — if they do, this points to a matching/normalization issue in the lookup itself; if they genuinely don't, this is a catalog-completeness gap requiring the product to be properly ingested, not a bug in this topic's triggering decision, which correctly identified that a check was warranted.

**Follow-up questions to expect:**

- "How would you extend this same 'blanket rule' pattern to other tools in this project's tool set?"
- "What would you measure to decide whether the blanket rule is worth refining into something more nuanced?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **This topic's blanket-rule simplification is a specific instance of a general engineering principle: when a nuanced judgment is hard to validate reliably, look for a simpler, more mechanically-detectable proxy trigger condition that captures most of the same value at much lower validation cost.** Recognizing this trade-off — accepting a small amount of unnecessary triggering in exchange for eliminating a harder-to-validate judgment — is a broadly applicable engineering decision-making pattern, not unique to this specific tool or case.
- **The two-step structure (mechanical detection, then a simple rule) mirrors Chapter 9 Topic 1's own layered triggering philosophy** — a cheap, reliable first step (detecting a named product) followed by a simple, deterministic second step (always check), rather than relying on a single, harder-to-validate judgment call for the entire decision at once.
- **This topic is the concrete, applied instance that makes Topic 4's Smart Saver FD test meaningful as a specific validation target** — without this topic's concrete decision rule already built, Topic 4's test would have nothing specific to validate; this topic's pattern is precisely what that test exists to confirm actually works correctly end to end.

### 10. Quick Revision Summary (for last-minute interview prep)

> This topic applies Topic 2's general confidence-based triggering principle to one specific, clean case: `lookup_product_catalog`. Rather than asking the model to judge its own confidence per product (a harder validation problem, per Topic 2), this pattern uses a simpler, more reliable blanket rule: detect whether a query mentions a specific, named product (a mechanical, pattern-matching step), and if so, always call `lookup_product_catalog` before stating any of that product's specific terms — regardless of how confident the model might otherwise feel. This is justified because product-specific figures are close to universally uncertain for a model to know reliably from training data (Chapter 9 Topic 8's argument), making the blanket rule's small, bounded cost (an occasional unnecessary lookup) clearly preferable to the alternative risk of confidently hallucinating a specific product's incorrect terms. This pattern adds no new retrieval mechanism — it reuses Chapter 10 Topic 6's `lookup_product_catalog` exactly as designed, contributing only the specific, narrow decision rule governing when that tool gets called. This is precisely the pattern Topic 4's Smart Saver FD test validates end to end.


### Module 1: The Two-Step Decision — Product Detection, Then a Blanket Trigger Rule

Implement the actual decision step: mechanical product-name detection, followed by a simple, reliable blanket rule -- no per-product confidence judgment needed.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: The two-step decision, implemented for real
# ------------------------------------------------------------------

PRODUCT_CATALOG = {
    "smart saver fd": {"interest_rate_percent": 7.65, "tenure_months": 18, "minimum_deposit_inr": 25000},
    "senior citizen fd": {"interest_rate_percent": 7.5, "tenure_months": 24, "minimum_deposit_inr": 10000},
}


def detect_named_product(query: str) -> str:
    """STEP 1: mechanical, pattern-matching detection -- does this
    query mention a specific, named product? Uses simple substring
    matching, including a check for partial/informal references."""
    text_lower = query.lower()
    for product_name in PRODUCT_CATALOG.keys():
        if product_name in text_lower:
            return product_name
        # check for a PARTIAL match (e.g. "smart saver" without "fd")
        name_words = product_name.replace(" fd", "").split()
        if all(word in text_lower for word in name_words) and len(name_words) > 1:
            return product_name
    return None


def should_check_catalog(query: str) -> bool:
    """STEP 2: the BLANKET rule -- if a named product was detected AT
    ALL, always check the catalog. NO per-product confidence judgment,
    exactly this topic's core simplification."""
    detected_product = detect_named_product(query)
    return detected_product is not None


TEST_QUERIES = [
    "What is the interest rate for the Smart Saver FD?",
    "Tell me about Smart Saver, please.",              # informal, partial reference
    "What is the general FD premature withdrawal policy?",  # no specific product named
    "Is Senior Citizen FD available for my mother?",
]

print("=" * 70)
print("MODULE 1: TWO-STEP DECISION -- DETECTION + BLANKET RULE")
print("=" * 70)

for query in TEST_QUERIES:
    detected = detect_named_product(query)
    should_check = should_check_catalog(query)
    print(f"\nQuery: '{query}'")
    print(f"  Detected product: {detected}")
    print(f"  Should check catalog? {should_check}")

print("\nNotice 'Tell me about Smart Saver, please.' -- an INFORMAL, partial")
print("reference -- still correctly triggers the check, demonstrating the")
print("detection step's robustness to real-world phrasing variation, not")
print("just exact catalog-name matches.")
print("\nModule 1 complete. Run Module 2 next.")


MODULE 1: TWO-STEP DECISION -- DETECTION + BLANKET RULE

Query: 'What is the interest rate for the Smart Saver FD?'
  Detected product: smart saver fd
  Should check catalog? True

Query: 'Tell me about Smart Saver, please.'
  Detected product: smart saver fd
  Should check catalog? True

Query: 'What is the general FD premature withdrawal policy?'
  Detected product: None
  Should check catalog? False

Query: 'Is Senior Citizen FD available for my mother?'
  Detected product: senior citizen fd
  Should check catalog? True

Notice 'Tell me about Smart Saver, please.' -- an INFORMAL, partial
reference -- still correctly triggers the check, demonstrating the
detection step's robustness to real-world phrasing variation, not
just exact catalog-name matches.

Module 1 complete. Run Module 2 next.


### Module 2: The Risk This Pattern Prevents — Confident, Unretrieved Wrong Answers

Simulate what happens WITHOUT this pattern (a model answering from general knowledge) versus WITH it (always checking first), using the REAL catalog data to reveal the concrete difference.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: The concrete risk this pattern prevents
# ------------------------------------------------------------------

def simulate_confident_unretrieved_answer(query: str) -> str:
    """Simulates what a model MIGHT confidently answer WITHOUT this
    pattern -- drawing on generic FD knowledge rather than the real
    catalog. Since Smart Saver FD's real terms were only just defined
    in this exercise (mirroring Chapter 9 Topic 8's argument), any
    confident answer here is NECESSARILY a plausible-sounding guess,
    not genuine knowledge."""
    return "Typically, FD products offer around 7 percent interest for a 12 to 24 month tenure."


def answer_with_catalog_check(query: str) -> dict:
    """The CORRECT pattern: check first, cite the REAL catalog data."""
    detected_product = detect_named_product(query)
    if detected_product is None:
        return {"used_catalog": False, "answer": "general policy answer (no specific product named)"}
    product_data = PRODUCT_CATALOG.get(detected_product)
    if product_data is None:
        return {"used_catalog": True, "answer": f"'{detected_product}' not found in catalog"}
    rate = product_data["interest_rate_percent"]
    tenure = product_data["tenure_months"]
    return {"used_catalog": True, "answer": f"{detected_product}: {rate} percent for {tenure} months"}


query = "What is the interest rate for the Smart Saver FD?"
ACTUAL_REAL_RATE = PRODUCT_CATALOG["smart saver fd"]["interest_rate_percent"]

print("=" * 70)
print("MODULE 2: CONFIDENT-UNRETRIEVED vs CATALOG-CHECKED ANSWER")
print("=" * 70)
print(f"Query: '{query}'")
print(f"ACTUAL real interest rate (from the catalog): {ACTUAL_REAL_RATE} percent\n")

unretrieved_answer = simulate_confident_unretrieved_answer(query)
print(f"WITHOUT this pattern (confident, unretrieved guess):")
print(f"  '{unretrieved_answer}'")
print(f"  Does this mention the ACTUAL rate ({ACTUAL_REAL_RATE})? "
      f"{str(ACTUAL_REAL_RATE) in unretrieved_answer}")

checked_result = answer_with_catalog_check(query)
checked_answer_text = checked_result["answer"]
print(f"\nWITH this pattern (catalog checked first):")
print(f"  '{checked_answer_text}'")
print(f"  Does this mention the ACTUAL rate ({ACTUAL_REAL_RATE})? "
      f"{str(ACTUAL_REAL_RATE) in checked_answer_text}")

print("\nCONFIRMED: the unretrieved guess states a GENERIC, WRONG figure (7 percent)")
print(f"instead of the real, specific rate ({ACTUAL_REAL_RATE} percent) -- exactly the")
print("confident-hallucination risk this topic's blanket rule is designed to")
print("eliminate for named-product queries specifically.")
print("\nModule 2 complete. Run Module 3 next.")


MODULE 2: CONFIDENT-UNRETRIEVED vs CATALOG-CHECKED ANSWER
Query: 'What is the interest rate for the Smart Saver FD?'
ACTUAL real interest rate (from the catalog): 7.65 percent

WITHOUT this pattern (confident, unretrieved guess):
  'Typically, FD products offer around 7 percent interest for a 12 to 24 month tenure.'
  Does this mention the ACTUAL rate (7.65)? False

WITH this pattern (catalog checked first):
  'smart saver fd: 7.65 percent for 18 months'
  Does this mention the ACTUAL rate (7.65)? True

CONFIRMED: the unretrieved guess states a GENERIC, WRONG figure (7 percent)
instead of the real, specific rate (7.65 percent) -- exactly the
confident-hallucination risk this topic's blanket rule is designed to
eliminate for named-product queries specifically.

Module 2 complete. Run Module 3 next.


### Module 3: Measuring the Pattern's Accuracy Across a Labeled Test Set

Run the full two-step decision against a labeled set covering named-product, informal-reference, and no-product cases, measuring real triggering accuracy.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: Measured accuracy across a labeled test set
# ------------------------------------------------------------------

LABELED_CATALOG_TRIGGER_SET = [
    ("What is the interest rate for the Smart Saver FD?", True),
    ("Tell me about Smart Saver, please.", True),                    # informal reference
    ("Senior Citizen FD minimum deposit?", True),
    ("What is the general FD withdrawal policy?", False),             # no specific product
    ("My loan EMI bounced this month.", False),                       # not FD-related at all
    ("Can I open a new FD account?", False),                          # FD-related, but no SPECIFIC product named
]

print("=" * 70)
print("MODULE 3: MEASURED ACCURACY -- TWO-STEP DECISION vs LABELED SET")
print("=" * 70)

correct = 0
for query, correct_answer in LABELED_CATALOG_TRIGGER_SET:
    decision = should_check_catalog(query)
    is_correct = decision == correct_answer
    correct += is_correct
    result_label = "CORRECT" if is_correct else "WRONG"
    print(f"\n'{query}'")
    print(f"  Correct answer: {correct_answer} | Decision: {decision} | {result_label}")

total = len(LABELED_CATALOG_TRIGGER_SET)
accuracy = correct / total * 100
print(f"\nOverall accuracy: {correct}/{total} = {accuracy:.0f}%")

print("\nThis measured accuracy is the concrete evidence this pattern actually")
print("works as designed -- reliably triggering for named-product mentions")
print("(exact OR informal) while correctly NOT triggering for general FD")
print("policy questions or genuinely unrelated topics.")

print("\nModule 3 complete. All Chapter 13 Topic 3 modules done.")
print()
print("=" * 70)
print("CHAPTER 13 TOPIC 3 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  Two SEPARATE steps: (1) mechanical detection of a named product
  mention, (2) a simple BLANKET rule -- no per-product confidence
  judgment needed, unlike Topic 2's more general, harder-to-validate
  approach.

  The detection step handles INFORMAL/partial references, not just
  exact catalog-name matches -- demonstrated concretely in Module 1.

  Demonstrated concretely (Module 2): an unretrieved, confident guess
  stated a WRONG generic rate (7 percent) instead of the real, specific
  rate -- exactly the risk this blanket rule prevents.

  This pattern reuses lookup_product_catalog (Chapter 10 Topic 6)
  UNCHANGED -- it only adds the DECISION of when to call it.

  This is exactly what Topic 4's Smart Saver FD test validates end
  to end.
""")


MODULE 3: MEASURED ACCURACY -- TWO-STEP DECISION vs LABELED SET

'What is the interest rate for the Smart Saver FD?'
  Correct answer: True | Decision: True | CORRECT

'Tell me about Smart Saver, please.'
  Correct answer: True | Decision: True | CORRECT

'Senior Citizen FD minimum deposit?'
  Correct answer: True | Decision: True | CORRECT

'What is the general FD withdrawal policy?'
  Correct answer: False | Decision: False | CORRECT

'My loan EMI bounced this month.'
  Correct answer: False | Decision: False | CORRECT

'Can I open a new FD account?'
  Correct answer: False | Decision: False | CORRECT

Overall accuracy: 6/6 = 100%

This measured accuracy is the concrete evidence this pattern actually
works as designed -- reliably triggering for named-product mentions
(exact OR informal) while correctly NOT triggering for general FD
policy questions or genuinely unrelated topics.

Module 3 complete. All Chapter 13 Topic 3 modules done.

CHAPTER 13 TOPIC 3 -- KEY POINTS TO REMEMBER

  